
# ML Failure Investigation Lab
## Thinking Like an ML Engineer

### Goal

In this workshop, you are NOT evaluated on:
- highest accuracy
- fastest implementation
- copying models

You ARE evaluated on:
- reasoning
- diagnosis
- justification
- investigation
- error analysis
- evidence-based improvements

---

## Rules

You must justify every decision.

For every modification:
1. What problem are you trying to solve?
2. What evidence suggests this problem exists?
3. Why should your solution help?
4. What tradeoff might it introduce?

---

## Main Question

> "Why is the model failing, and how can we prove it?"



# Dataset

We will use the Credit Card Fraud Detection dataset.

Download manually from:

https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

Place `creditcard.csv` in the same folder as this notebook.


In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt



# Task 1 — Load and Investigate the Dataset

Without training any model yet:

Answer:
## 1. How many rows and columns exist? rowns: 284807, ncolumn: 31
## 2. Are there missing values? No Zero missing
## 4. Is the dataset balanced? No, Data is high imbalance
## 5. What immediate risks do you notice? Data is imbalance Class 0    0.998273 1    0.001727


## Reasoning Questions

## 1. Why might class imbalance be dangerous? it make model learning very difficult and predict accuracy 99 but always say 0
## 2. Why can accuracy become misleading? 99 but always say 0 care only about number of true
## 3. If fraud cases are very rare, what metric may matter more? percision or recall based on what you care, macro avg F1 score is good for both


In [2]:

df = pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")

df.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
df.shape

(284807, 31)

In [4]:

df.info()

df.isnull().sum()

df["Class"].value_counts()

df["Class"].value_counts(normalize=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64


# Accuracy Trap

Suppose:
- 99.8% of transactions are NOT fraud

A model predicting:
> "Not Fraud" for every sample

could still achieve extremely high accuracy.

---

## Formula

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

---

## Think Carefully

## Would this model actually be useful? XGBoost, LightGMB

Why or why not? 
## this model is Decision trees + Boosting every new tree foucus on past error 
## Can use class_weights
## Good for nonlinear pattarns


# Task 2 — Train/Test Split

Questions:
## 1. Why do we use stratify? make sure split is same percent in train and test important for imbalnce no all 0 in train
## 2. What risk happens if fraud samples are distributed unevenly? train is all 0 no var model will learn all is 0


In [5]:

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



# Task 3 — Scaling Decision

We will scale features before Logistic Regression.

## Questions

## 1. Why might scaling matter for Logistic Regression? Function approximation, Gradiant Descent be very long time, convergence be slow
## 2. Would scaling matter equally for Decision Trees? No it work with split
## 3. Which algorithms are sensitive to feature magnitude? Model see large is more important


In [6]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



# Task 4 — Baseline Model

Train a Logistic Regression model WITHOUT solving imbalance.

Before running:
## - Predict what might happen ALL test is 0
## - Which metric may look deceptively good? Recall and macro F1


In [7]:

model = LogisticRegression()

model.fit(X_train_scaled, y_train)

preds = model.predict(X_test_scaled)


In [8]:

print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds))
print("Recall:", recall_score(y_test, preds))
print("F1:", f1_score(y_test, preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))


Accuracy: 0.9991397773954567
Precision: 0.8266666666666667
Recall: 0.6326530612244898
F1: 0.7167630057803468

Confusion Matrix
[[56851    13]
 [   36    62]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.63      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962




# Task 5 — Error Analysis

Analyze the results carefully.

---

## Questions

## 1. Is the model actually good? No class 0 is 0.998 accuracy is 0.999 might overfiting all 0, recall is low 0.63
## 2. Which metric is most concerning? recall
## 3. What type of errors are dangerous here? FN model say is normal but it's not normal
## 4. What does the confusion matrix reveal? TP,TN,FN,FP what model predict wrong in
## 5. Would this model be acceptable in banking? NO always normal 0

---

## Recall Formula

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

---

## Important Thinking Question

What happens if:
## - False Negatives increase? model say normal but it fraud loss money

In fraud detection:
- which is worse:
  - false positives?
  ## - false negatives? most important 

Why?
## model say normal but it fraud loss money


# Task 6 — Improve the Model

You may choose ONLY ONE improvement initially.

Choose ONE:
- class_weight
## - resampling i will use undersampling and oversampling
- threshold tuning
- feature engineering
- different model

---

## Before Coding

You MUST justify:
## 1. Why this improvement targets the failure Make percent more close than 0.998
## 2. What evidence supports your choice Make percent more close than 0.998 
## 3. What tradeoff might appear improve class balance but change the data distribution 



# Hint

Logistic Regression supports:

```python
class_weight="balanced"
```

But:

DO NOT use it blindly.

First explain:
## - what it changes mathematically make minor more expensive and majority less important 
## - why it may help recall change what model cares about
## - what downside it may introduce more FP percision drop


In [9]:

balanced_model = LogisticRegression(class_weight="balanced")

balanced_model.fit(X_train_scaled, y_train)

balanced_preds = balanced_model.predict(X_test_scaled)


In [10]:

print("Accuracy:", accuracy_score(y_test, balanced_preds))
print("Precision:", precision_score(y_test, balanced_preds))
print("Recall:", recall_score(y_test, balanced_preds))
print("F1:", f1_score(y_test, balanced_preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, balanced_preds))

print("\nClassification Report")
print(classification_report(y_test, balanced_preds))


Accuracy: 0.9755275446789088
Precision: 0.06097560975609756
Recall: 0.9183673469387755
F1: 0.11435832274459974

Confusion Matrix
[[55478  1386]
 [    8    90]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962




# Task 7 — Tradeoff Investigation

Compare:
## - baseline model  recall is 0.63
## - balanced model  No care only on recall bad precision

---

## Questions

## 1. Which metric improved most? recall
## 2. Which metric became worse? precision
## 3. Why did this happen? model
## 4. Is higher recall worth lower precision? more FP percision drop
## 5. Would business context affect this decision? model care about FN no loss money but UX problem model say fraud more 

---

## Critical Thinking

## Should ALL fraud systems maximize recall aggressively? NO

Why or why not?
## make new problem model say you fraud but you not 


# Task 8 — Analyze Failure Cases

Investigate:
- False Positives
- False Negatives

---

## Questions

## 1. Do failed samples share patterns? Failed samples often look very similar to normal transactions, making them hard to separate
## 2. Are some transactions consistently difficult? Yes some transactions stay difficult because they contain both normal and suspicious behavior
## 3. Which error type remains hardest? False Negatives are hardest because some fraud transactions look normal to the model


In [11]:

results = X_test.copy()

results["Actual"] = y_test.values
results["Predicted"] = balanced_preds

false_negatives = results[
    (results["Actual"] == 1) &
    (results["Predicted"] == 0)
]

false_negatives.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V22,V23,V24,V25,V26,V27,V28,Amount,Actual,Predicted
50537,44532.0,-0.234922,0.355413,1.972183,-1.255593,-0.681387,-0.665732,0.059110,-0.003153,1.122451,...,0.912107,-0.286338,0.451208,0.188315,-0.531846,0.123185,0.039581,1.00,1,0
157585,110087.0,1.934946,0.650678,-0.286957,3.987828,0.316052,-0.099449,-0.021483,-0.172327,0.508730,...,-0.190974,0.219976,-0.216597,-0.136692,-0.129954,-0.050077,-0.051082,1.00,1,0
96341,65728.0,1.227614,-0.668974,-0.271785,-0.589440,-0.604795,-0.350285,-0.486365,-0.010809,-0.794944,...,-0.295255,-0.180459,-0.436539,0.494649,-0.283738,-0.001128,0.035075,98.01,1,0
68067,52814.0,-1.101847,-1.632441,0.901067,0.847753,-1.249091,0.654937,1.448868,0.023308,-0.136742,...,0.835795,1.179955,-0.029091,-0.300896,0.699175,-0.336072,-0.177587,519.90,1,0
219025,141565.0,0.114965,0.766762,-0.494132,0.116772,0.868169,-0.477982,0.438496,0.063073,-0.186207,...,-0.706865,0.131405,0.600742,-0.604264,0.262938,0.099145,0.010810,4.49,1,0



# Bonus Challenge 1 — Threshold Tuning

Try:
## - 0.5 threshold 
## - 0.3 threshold 
## - 0.1 threshold 

Then analyze:
- precision vs recall tradeoff
## - 0.5 threshold higher precision but lower recall because the model predicts fraud more carefully
## - 0.3 threshold recall increased to 0.918, meaning most fraud cases were detected, but precision dropped to 0.027 because many normal transactions were falsely flagged
## - 0.1 threshold recall would likely increase even more, but precision would decrease further since the model would classify many more transactions as fraud

In [14]:

probs = balanced_model.predict_proba(X_test_scaled)[:, 1]

threshold = 0.1

custom_preds = (probs >= threshold).astype(int)

print("Precision:", precision_score(y_test, custom_preds))
print("Recall:", recall_score(y_test, custom_preds))
print("F1:", f1_score(y_test, custom_preds))


Precision: 0.008152173913043478
Recall: 0.9489795918367347
F1: 0.016165478880584044



# Bonus Challenge 2 — Random Forest Comparison

Questions:
## 1. Why might Random Forest behave differently? behaves differently because it uses many decision trees and split rules instead of learning one linear boundary like logistic regression
## 2. Why may scaling matter less? Scaling matters less because tree models compare feature values using thresholds(Split), not distances or gradient size
## 3. Which model would you trust more here? I would trust XGBoost or weighted logistic regression more random forest can all tree vote say 0


In [15]:

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_preds))
print("Precision:", precision_score(y_test, rf_preds))
print("Recall:", recall_score(y_test, rf_preds))
print("F1:", f1_score(y_test, rf_preds))


Accuracy: 0.9995962220427653
Precision: 0.9411764705882353
Recall: 0.8163265306122449
F1: 0.8743169398907104



# Bonus Challenge 3 — Leakage Trap

Read the code carefully before running.

Questions:
## 1. Why might this create suspiciously high performance? The model may get extremely high accuracy because the Leakage feature directly contains information from the target Class
## 2. What kind of leakage is this? his is target leakage because the input features include information from the actual label
## 3. Why is leakage dangerous in production systems? Less generization Leakage is dangerous because the model learns hidden answers unavailable in real production 


In [17]:

df["Leakage"] = df["Class"] + np.random.normal(0, 0.01, len(df))

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression()

model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))


Accuracy: 0.9990695551420246


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



# Final Reflection

Answer carefully.

---

## Reflection Questions

1. What was the REAL problem in this dataset? The real problem was extreme class imbalance where fraud transactions were very rare
2. Why was accuracy misleading? Accuracy was misleading because the model could predict almost all transactions as normal and still get very high accuracy
3. What evidence proved imbalance existed? The class distribution and very low fraud detection performance showed strong imbalance
4. Why did the second model improve recall? The second model improved recall because it gave more importance to fraud samples using imbalance handling methods
5. What tradeoff was introduced? The tradeoff was lower precision and more false positives
6. What would you deploy in production? I would deploy a weighted boosting model with threshold tuning and continuous monitoring
7. What additional experiments would you run? I would test different thresholds, class weights, resampling methods, feature engineering, and time-based validation
8. What assumptions might still be dangerous? Dangerous assumptions include assuming future fraud patterns stay the same and assuming training data fully represents real-world behavior

---

# Final Engineering Question

> "How do you know your improvement genuinely solved the problem?"
> 1. Test on true unseen data (new data)
> 2. Check fraud-specific metrics (not accuracy)
> 3. Compare confusion matrices
> 4. Stability across thresholds
> 5. Business cost evaluation
> 6. Leakage and bias audit
>    
> An improvement is REAL only if it improves out-of-sample fraud detection under realistic time + cost constraints, not just offline metrics



# Grading Rubric

You are graded on:

- reasoning quality
- justification quality
- investigation depth
- evidence usage
- interpretation
- tradeoff understanding

NOT only final metrics.
